In [3]:
import pandas as pd
import numpy as np

ltdc = pd.read_csv("similarity_outputs/copula_ltdc_matrix.csv", index_col=0)
utdc = pd.read_csv("similarity_outputs/copula_utdc_matrix.csv", index_col=0)

N = ltdc.shape[0]
idx = np.triu_indices(N, 1)

l = ltdc.values[idx]
u = utdc.values[idx]

# Align: both must be non-zero (i.e. pair was computed in both matrices)
mask = (l > 0) & (u > 0)
l = l[mask]
u = u[mask]

print(f"Valid pairs (in both matrices): {mask.sum()}")
print(f"LTDC > UTDC (crash > surge):    {(l > u).sum()} / {len(l)}  ({(l>u).mean()*100:.1f}%)")

Valid pairs (in both matrices): 180163
LTDC > UTDC (crash > surge):    157639 / 180163  (87.5%)


In [4]:
import pandas as pd
import numpy as np

hmm = pd.read_csv("similarity_outputs/hmm_similarity_matrix.csv", index_col=0)
vals = hmm.values[np.triu_indices(hmm.shape[0], 1)]
print(f"non-nan: {(~np.isnan(vals)).sum()}")
print(f"mean:    {np.nanmean(vals):.4f}")
print(f"median:  {np.nanmedian(vals):.4f}")

for k in range(3):
    rc = pd.read_csv(f"similarity_outputs/hmm_regime_{k}_corr.csv", index_col=0)
    v = rc.values[np.triu_indices(rc.shape[0], 1)]
    print(f"Regime {k} — non-nan pairs: {(~np.isnan(v)).sum()}  median: {np.nanmedian(v):.4f}")

non-nan: 188805
mean:    0.6358
median:  0.7304
Regime 0 — non-nan pairs: 181996  median: 0.6162
Regime 1 — non-nan pairs: 182480  median: 0.3468
Regime 2 — non-nan pairs: 181013  median: 0.8424


In [5]:
import pandas as pd
import numpy as np

pc = pd.read_csv("similarity_outputs/glasso_partial_corr_matrix.csv", index_col=0)
gs = pd.read_csv("similarity_outputs/glasso_similarity_matrix.csv", index_col=0)

N = gs.shape[0]
idx = np.triu_indices(N, 1)

v = gs.values[idx]
print(f"GLASSO sim — mean: {np.nanmean(v):.4f}  median: {np.nanmedian(v):.4f}  std: {np.nanstd(v):.4f}")

pc_v = pc.values[idx]
print(f"Partial corr — mean: {np.nanmean(pc_v):.4f}  std: {np.nanstd(pc_v):.4f}")
print(f"Non-zero edges (|pc| > 0.01): {(np.abs(pc_v) > 0.01).sum()}")
print(f"Negative partial corr pairs:  {(pc_v < -0.01).sum()}")
print(f"Positive partial corr pairs:  {(pc_v > 0.01).sum()}")

GLASSO sim — mean: 0.5002  median: 0.5000  std: 0.0030
Partial corr — mean: 0.0004  std: 0.0059
Non-zero edges (|pc| > 0.01): 2430
Negative partial corr pairs:  256
Positive partial corr pairs:  2174


In [6]:
dtw = pd.read_csv("similarity_outputs/dtw_similarity_matrix.csv", index_col=0)
N = dtw.shape[0]
idx = np.triu_indices(N, 1)
v = dtw.values[idx]
print(f"DTW sim — mean: {np.nanmean(v):.4f}  median: {np.nanmedian(v):.4f}  std: {np.nanstd(v):.4f}")
print(f"High similarity (>0.8): {(v>0.8).sum()} / {len(v)}")
print(f"Low similarity (<0.2):  {(v<0.2).sum()} / {len(v)}")

DTW sim — mean: 0.3764  median: 0.3679  std: 0.0432
High similarity (>0.8): 16 / 188805
Low similarity (<0.2):  0 / 188805


In [7]:
import pandas as pd
import numpy as np

coint = pd.read_csv("similarity_outputs/coint_similarity_matrix.csv", index_col=0)
stats = pd.read_csv("similarity_outputs/coint_pairs_table.csv")

N = coint.shape[0]
idx = np.triu_indices(N, 1)
v = coint.values[idx]

print(f"Coint sim — mean: {np.nanmean(v):.4f}  median: {np.nanmedian(v):.4f}  std: {np.nanstd(v):.4f}")
print(f"Cointegrated pairs: {len(stats)}")
print(f"Total pairs: {len(v)}")
print(f"Cointegration rate: {len(stats)/len(v)*100:.2f}%")
print(f"Strong pairs (|ADF| > 4): {(stats['adf_stat'].abs() > 4).sum()}")
print(f"Mean ADF stat: {stats['adf_stat'].mean():.4f}")
print(f"Mean beta: {stats['beta'].mean():.4f}")
print(f"Beta > 0 (same direction): {(stats['beta'] > 0).sum()} / {len(stats)}")
print(f"Beta range: {stats['beta'].min():.4f} to {stats['beta'].max():.4f}")

Coint sim — mean: 0.0105  median: 0.0103  std: 0.0067
Cointegrated pairs: 60608
Total pairs: 188805
Cointegration rate: 32.10%
Strong pairs (|ADF| > 4): 13053
Mean ADF stat: -3.7457
Mean beta: 0.8248
Beta > 0 (same direction): 55568 / 60608
Beta range: -988.8254 to 1994.1765


In [5]:
import polars as pl
df = pl.scan_parquet("../data/crypto_with_indicators.parquet").collect()
df = df.filter(pl.col("is_stablecoin") == False)
df = df.with_columns(pl.col("date").cast(pl.Date))
df = df.filter(pl.col("date").dt.year() > 2018)
print(f"Coins : {df['coin_id'].n_unique()}")
print(f"Range : {df['date'].min()} → {df['date'].max()}")
print(f"Rows  : {len(df):,}")

Coins : 698
Range : 2021-04-20 → 2026-04-17
Rows  : 661,442


In [7]:
import polars as pl
df = pl.scan_parquet("../data/crypto_with_indicators.parquet").collect()

# Option 1 — unique coins with symbol and name
coins = df.select(['coin_id', 'symbol', 'name']).unique().sort('name')

print(f"\nTotal unique coins: {coins.height}")
df.select(['coin_id', 'symbol', 'name']).unique().sort('name').write_csv('coin_list.csv')



Total unique coins: 715


In [9]:
import csv

data = [
("zero-gravity","0G","0G","Infrastructure"),
("0x","ZRX","0x Protocol","DeFi"),
("1inch","1INCH","1INCH","DeFi"),
("newton-project","AB","AB",""),
("adi-token","ADI","ADI",""),
("ai-rig-complex","ARC","AI Rig Complex","AI"),
("apenft","NFT","AINFT","Gaming"),
("aioz-network","AIOZ","AIOZ Network","Infrastructure"),
("aleo","ALEO","ALEO","Privacy|Layer 1"),
("airtor-protocol","ANYONE","ANyONe Protocol","Privacy|Infrastructure"),
("ark","ARK","ARK","Layer 1"),
("arpa","ARPA","ARPA","Privacy|Infrastructure"),
("as-roma-fan-token","ASR","AS Roma Fan Token",""),
("agora-dollar","AUSD","AUSD","Payments"),
("concierge-io","AVA","AVA (Travala)","Payments"),
("stp-network","AWE","AWE Network","Infrastructure"),
("aave","AAVE","Aave","DeFi|Liquid staking"),
("access-protocol","ACS","Access Protocol","Infrastructure"),
("achain","ACT","Achain","Layer 1"),
("across-protocol","ACX","Across Protocol","Layer 2|Infrastructure"),
("act-i-the-ai-prophecy","ACT","Act I The AI Prophecy","AI|Meme"),
("acurast","ACU","Acurast","Infrastructure"),
("adventure-gold","AGLD","Adventure Gold","Gaming"),
("aergo","AERGO","Aergo","Layer 1"),
("aerodrome-finance","AERO","Aerodrome Finance","DeFi"),
("aethir","ATH","Aethir","AI|Infrastructure"),
("aevo-exchange","AEVO","Aevo Exchange","DeFi|Exchange token"),
("akash-network","AKT","Akash Network","Infrastructure|AI"),
("akedo","AKE","Akedo","Gaming"),
("alchemist-ai","ALCH","Alchemist AI","AI"),
("alchemix","ALCX","Alchemix","DeFi"),
("alchemy-pay","ACH","Alchemy Pay","Payments"),
("algorand","ALGO","Algorand","Layer 1"),
("alien-worlds","TLM","Alien Worlds","Gaming"),
("allora","ALLO","Allora","AI"),
("alpha-fi","ALPHA","Alpha Fi","DeFi"),
("altlayer","ALT","AltLayer","Layer 2"),
("amp-token","AMP","Amp","Payments"),
("anchored-coins-eur","AEUR","Anchored Coins AEUR","Real-world assets|Payments"),
("anime","ANIME","Animecoin","Gaming|Meme"),
("ankr","ANKR","Ankr Network","Infrastructure|Liquid staking"),
("anoma","XAN","Anoma","Privacy|Layer 1"),
("anyspend","ANY","Anyspend","Payments"),
("apecoin","APE","ApeCoin","Gaming|Meme"),
("api3","API3","Api3","Oracle"),
("apro","AT","Apro","Oracle"),
("aptos","APT","Aptos","Layer 1"),
("apu-s-club","APU","Apu Apustaja","Meme"),
("arbitrum","ARB","Arbitrum","Layer 2"),
("ardor","ARDR","Ardor","Layer 1"),
("arkham","ARKM","Arkham","Infrastructure"),
("fetch-ai","FET","Artificial Superintelligence Alliance","AI"),
("arweave","AR","Arweave","Infrastructure"),
("astar","ASTR","Astar","Layer 1|Layer 2"),
("aster-2","ASTER","Aster","DeFi"),
("atletico-madrid","ATM","Atletico Madrid Fan Token",""),
("audius","AUDIO","Audius","Infrastructure"),
("augur","REP","Augur","Oracle|DeFi"),
("aurora-near","AURORA","Aurora","Layer 2"),
("automata","ATA","Automata","Infrastructure"),
("autonomi","ANT","Autonomi","Infrastructure"),
("ava-ai","AVA","Ava AI","AI"),
("avail","AVAIL","Avail","Infrastructure|Layer 2"),
("avalanche-2","AVAX","Avalanche","Layer 1"),
("avalaunch","XAVA","Avalaunch","Infrastructure"),
("avantis","AVNT","Avantis","DeFi"),
("axelar","AXL","Axelar","Infrastructure|Layer 2"),
("axie-infinity","AXS","Axie Infinity","Gaming"),
("aztec","AZTEC","Aztec","Privacy|Layer 2"),
("b3","B3","B3 (Base)","Layer 2"),
("benqi","QI","BENQI","DeFi|Liquid staking"),
("bfusd","BFUSD","BFUSD","Payments"),
("binancecoin","BNB","BNB","Exchange token|Layer 1"),
("bob-build-on-bitcoin","BOB","BOB (Build on Bitcoin)","Layer 2"),
("book-of-meme","BOME","BOOK OF MEME","Meme"),
("bsquared-network","B2","BSquared Network","Layer 2"),
("binance-usd","BUSD","BUSD","Payments"),
("baby-doge-coin","BABYDOGE","Baby Doge Coin","Meme"),
("baby-shark-universe","BSU","Baby Shark Universe","Gaming|Meme"),
("babylon","BABY","Babylon","Infrastructure|Liquid staking"),
("badger-dao","BADGER","Badger","DeFi"),
("balancer","BAL","Balancer","DeFi"),
("banana-for-scale-2","BANANAS31","Banana For Scale","Meme"),
("banana-gun","BANANA","Banana Gun","DeFi|Infrastructure"),
("bancor","BNT","Bancor Network","DeFi"),
("band-protocol","BAND","Band","Oracle"),
("basic-attention-token","BAT","Basic Attention","Payments"),
("beam-2","BEAM","Beam","Gaming|Layer 1"),
("bedrock-token","BR","Bedrock","Liquid staking"),
("beldex","BDX","Beldex","Privacy"),
("bella-protocol","BEL","Bella Protocol","DeFi"),
("berachain-bera","BERA","Berachain","Layer 1"),
("bertram-the-pomeranian","BERT","Bertram The Pomeranian","Meme"),
("beta-finance","BETA","Beta Finance","DeFi"),
("biconomy","BICO","Biconomy","Infrastructure"),
("big-time","BIGTIME","Big Time","Gaming"),
("bio-protocol","BIO","Bio Protocol","Real-world assets"),
("bitmart-token","BMX","BitMart","Exchange token"),
("bittorrent","BTT","BitTorrent","Infrastructure"),
("bitcoin","BTC","Bitcoin","Layer 1|Payments"),
("bitcoin-cash","BCH","Bitcoin Cash","Layer 1|Payments"),
("bitcoin-gold","BTG","Bitcoin Gold","Layer 1"),
("bitrise-token","BRISE","Bitgert","Layer 1"),
("bitget-token","BGB","Bitget Token","Exchange token"),
("bitlayer-bitvm","BTR","Bitlayer","Layer 2"),
("bittensor","TAO","Bittensor","AI"),
("blast","BLAST","Blast","Layer 2"),
("bless-2","BLESS","Bless","Infrastructure"),
("bluefin","BLUE","Bluefin","DeFi"),
("blur","BLUR","Blur","DeFi"),
("bluwhale","BLUAI","Bluwhale","AI"),
("boba-network","BOBA","Boba Network","Layer 2"),
("bonfida","FIDA","Bonfida","DeFi|Infrastructure"),
("bonk","BONK","Bonk","Meme"),
("auction","AUCTION","Bounce","DeFi"),
("bouncebit","BB","BounceBit","Layer 1|DeFi"),
("boundless","ZKC","Boundless","Infrastructure"),
("based-brett","BRETT","Brett","Meme"),
("brevis","BREV","Brevis","Infrastructure"),
("build-on-bnb","BOB","Build On BNB","Infrastructure"),
("carv","CARV","CARV","Gaming|AI"),
("cash-4","CASH","CASH","Payments"),
("coti","COTI","COTI","Payments|Layer 1"),
("cyberconnect","CYBER","CYBER","Infrastructure"),
("caldera","ERA","Caldera","Layer 2|Infrastructure"),
("canton-network","CC","Canton","Infrastructure"),
("cardano","ADA","Cardano","Layer 1"),
("cartesi","CTSI","Cartesi","Layer 2"),
("casper-network","CSPR","Casper Network","Layer 1"),
("catizen","CATI","Catizen","Gaming|Meme"),
("celer-network","CELR","Celer Network","Layer 2|Infrastructure"),
("celestia","TIA","Celestia","Infrastructure"),
("celo","CELO","Celo","Layer 1|Payments"),
("cetus-protocol","CETUS","Cetus Protocol","DeFi"),
("chaingpt","CGPT","ChainGPT","AI"),
("chainbase","C","Chainbase","Infrastructure"),
("chainflip","FLIP","Chainflip","DeFi|Infrastructure"),
("chainlink","LINK","Chainlink","Oracle"),
("checkmate-2","CHECK","Checkmate",""),
("cheems-token","CHEEMS","Cheems Token","Meme"),
("chia","XCH","Chia","Layer 1"),
("chiliz","CHZ","Chiliz","Infrastructure"),
("chex-token","CHEX","Chintai","Real-world assets"),
("chromaway","CHR","Chromia","Layer 1"),
("civic","CVC","Civic","Infrastructure"),
("clearpool","CPOOL","Clearpool","DeFi|Real-world assets"),
("sanctum-2","CLOUD","Cloud","Liquid staking"),
("cow-protocol","COW","CoW Protocol","DeFi"),
("codatta","XNY","Codatta","Infrastructure"),
("coin98","C98","Coin98","DeFi|Payments"),
("comedian","BAN","Comedian","Meme"),
("compound-governance-token","COMP","Compound","DeFi"),
("concordium","CCD","Concordium","Layer 1"),
("conflux-token","CFX","Conflux","Layer 1"),
("constellation-labs","DAG","Constellation","Layer 1"),
("constitutiondao","PEOPLE","ConstitutionDAO",""),
("contentos","COS","Contentos","Infrastructure"),
("convex-finance","CVX","Convex Finance","DeFi|Liquid staking"),
("cookie","COOKIE","Cookie DAO","AI"),
("coq-inu","COQ","Coq Inu","Meme"),
("corn-3","CORN","Corn","Layer 2"),
("cosmos","ATOM","Cosmos Hub","Layer 1|Infrastructure"),
("creditcoin-2","CTC","Creditcoin","Real-world assets|Layer 1"),
("crypto-com-chain","CRO","Cronos","Exchange token|Layer 1"),
("curve-dao-token","CRV","Curve DAO","DeFi"),
("dai-on-pulsechain","DAI","DAI on PulseChain","Payments"),
("dao-maker","DAO","DAO Maker","Infrastructure"),
("dia-data","DIA","DIA","Oracle"),
("dodo","DODO","DODO","DeFi"),
("dusk-network","DUSK","DUSK","Privacy|Layer 1"),
("dai","DAI","Dai","Payments|DeFi"),
("dash","DASH","Dash","Payments|Privacy"),
("dexe","DEXE","DeXe","DeFi|AI"),
("decentraland","MANA","Decentraland","Gaming"),
("decred","DCR","Decred","Layer 1|Privacy"),
("deep","DEEP","DeepBook","DeFi"),
("definitive","EDGE","Definitive","DeFi"),
("degen-base","DEGEN","Degen","Meme"),
("dent","DENT","Dent","Payments"),
("derive","DRV","Derive","DeFi"),
("digibyte","DGB","DigiByte","Layer 1"),
("dimitra","DMTR","Dimitra","Real-world assets"),
("dog-go-to-the-moon-rune","DOG","Dog (Bitcoin)","Meme"),
("dogecoin","DOGE","Dogecoin","Meme|Payments"),
("dogelon-mars","ELON","Dogelon Mars","Meme"),
("dogs-2","DOGS","Dogs","Meme"),
("dolomite","DOLO","Dolomite","DeFi"),
("doublezero","2Z","DoubleZero","Infrastructure"),
("drift-protocol","DRIFT","Drift Protocol","DeFi"),
("dymension","DYM","Dymension","Layer 1|Infrastructure"),
("ecomi","OMI","ECOMI","Gaming"),
("ethgas-2","GWEI","ETHGas","Infrastructure"),
("euro-coin","EURC","EURC","Payments|Real-world assets"),
("schuman-europ","EUROP","EURØP","Payments|Real-world assets"),
("echelon-prime","PRIME","Echelon Prime","Gaming"),
("eclipse-3","ES","Eclipse","Layer 2"),
("eigenlayer","EIGEN","EigenCloud (prev. EigenLayer)","Infrastructure|Liquid staking"),
("elastos","ELA","Elastos","Infrastructure"),
("electroneum","ETN","Electroneum","Payments"),
("ellipsis","EPS","Ellipsis [OLD]","DeFi"),
("energy-web-token","EWT","Energy Web Token","Real-world assets"),
("enjincoin","ENJ","Enjin Coin","Gaming"),
("enso","ENSO","Enso","DeFi"),
("melon","MLN","Enzyme","DeFi"),
("epic-chain","EPIC","Epic Chain","Layer 1"),
("epic-cash","EPIC","Epic Private Internet Cash","Privacy|Payments"),
("ergo","ERG","Ergo","Layer 1|Privacy"),
("espresso","ESP","Espresso","Infrastructure"),
("ethena","ENA","Ethena","DeFi"),
("ethena-usde","USDE","Ethena USDe","Payments|DeFi"),
("ether-fi","ETHFI","Ether.fi","Liquid staking"),
("ethereum","ETH","Ethereum","Layer 1"),
("ethereum-classic","ETC","Ethereum Classic","Layer 1"),
("ethereum-name-service","ENS","Ethereum Name Service","Infrastructure"),
("ethereum-pow-iou","ETHW","EthereumPoW","Layer 1"),
("euler","EUL","Euler","DeFi"),
("eurite","EURI","Eurite","Payments|Real-world assets"),
("fc-barcelona-fan-token","BAR","FC Barcelona Fan Token",""),
("fc-porto","PORTO","FC Porto",""),
("fight-2","FIGHT","FIGHT","Gaming"),
("flock-2","FLOCK","FLOCK","AI"),
("floki","FLOKI","FLOKI","Meme"),
("folks","FOLKS","FOLKS","DeFi"),
("falcon-finance-ff","FF","Falcon Finance","DeFi"),
("fartcoin","FARTCOIN","Fartcoin","Meme"),
("fidelity-digital-dollar","FIDD","Fidelity Digital Dollar","Payments|Real-world assets"),
("filecoin","FIL","Filecoin","Infrastructure"),
("zcoin","FIRO","Firo","Privacy|Layer 1"),
("first-digital-usd","FDUSD","First Digital USD","Payments"),
("flare-networks","FLR","Flare","Oracle|Layer 1"),
("flow","FLOW","Flow","Layer 1|Gaming"),
("zelcash","FLUX","Flux","Infrastructure"),
("fogo","FOGO","Fogo","Layer 1"),
("forta","FORT","Forta","Infrastructure"),
("four","FORM","Four","DeFi"),
("fractal-bitcoin","FB","Fractal Bitcoin","Layer 2"),
("frax-share","FRAX","Frax (prev. FXS)","DeFi"),
("fuel-network","FUEL","Fuel Network","Layer 2"),
("endurance","ACE","Fusionist","Gaming"),
("project-galaxy","GAL","GAL (migrated to Gravity - G)","Infrastructure"),
("gala","GALA","GALA","Gaming"),
("stepn","GMT","GMT","Gaming"),
("gmx","GMX","GMX","DeFi"),
("griffain","GRIFFAIN","GRIFFAIN","AI|Meme"),
("gains-network","GNS","Gains Network","DeFi"),
("galatasaray-fan-token","GAL","Galatasaray Fan Token",""),
("gas","GAS","Gas","Infrastructure"),
("gigachad-2","GIGA","Gigachad","Meme"),
("giggle-fund","GIGGLE","Giggle Fund","Meme"),
("gitcoin","GTC","Gitcoin","Infrastructure"),
("global-dollar","USDG","Global Dollar","Payments"),
("gnosis","GNO","Gnosis","Infrastructure|DeFi"),
("gmt-token","GOMINING","GoMining Token","Infrastructure"),
("goplus-security","GPS","GoPlus Security","Infrastructure"),
("goatseus-maximus","GOAT","Goatseus Maximus","AI|Meme"),
("gods-unchained","GODS","Gods Unchained","Gaming"),
("goldfinch","GFI","Goldfinch","DeFi|Real-world assets"),
("golem","GLM","Golem","Infrastructure|AI"),
("grass","GRASS","Grass","AI|Infrastructure"),
("g-token","G","Gravity (by Galxe)","Infrastructure"),
("gunz","GUN","Gunz","Gaming"),
("home","HOME","HOME",""),
("htx-dao","HTX","HTX DAO","Exchange token"),
("haedal","HAEDAL","Haedal Protocol","Liquid staking"),
("hamster-kombat","HMSTR","Hamster Kombat","Gaming|Meme"),
("harmony","ONE","Harmony","Layer 1"),
("harvest-finance","FARM","Harvest Finance","DeFi"),
("hashkey-ecopoints","HSK","HashKey Platform Token","Exchange token"),
("hashflow","HFT","Hashflow","DeFi"),
("hedera-hashgraph","HBAR","Hedera","Layer 1"),
("hegic","HEGIC","Hegic","DeFi"),
("heima","HEI","Heima","Layer 1"),
("helium","HNT","Helium","Infrastructure"),
("hemi","HEMI","Hemi","Layer 2"),
("heroes-of-mavia","MAVIA","Heroes of Mavia","Gaming"),
("heyanon","ANON","Hey Anon","AI"),
("hive","HIVE","Hive","Layer 1"),
("hivemapper","HONEY","Hivemapper","Infrastructure|Real-world assets"),
("holotoken","HOT","Holo","Infrastructure"),
("holoworld","HOLO","Holoworld","AI|Gaming"),
("honey-3","HONEY","Honey","DeFi"),
("zencash","ZEN","Horizen","Privacy|Layer 1"),
("huma-finance","HUMA","Huma Finance","DeFi|Real-world assets"),
("humanity","H","Humanity","Infrastructure"),
("hydradx","HDX","Hydration","DeFi"),
("hyperlane","HYPER","Hyperlane","Infrastructure"),
("hyperliquid","HYPE","Hyperliquid","DeFi|Exchange token"),
("icon","ICX","ICON","Layer 1"),
("infinit","IN","INFINIT","DeFi"),
("iostoken","IOST","IOST","Layer 1"),
("iota","IOTA","IOTA","Layer 1|Infrastructure"),
("everipedia","IQ","IQ","AI"),
("iagon","IAG","Iagon","Infrastructure"),
("illuvium","ILV","Illuvium","Gaming"),
("immutable-x","IMX","Immutable","Layer 2|Gaming"),
("impossible-cloud-network-token","ICNT","Impossible Cloud Network Token","Infrastructure"),
("infinex-2","INX","Infinex","DeFi|Exchange token"),
("infinity-ground","AIN","Infinity Ground","AI"),
("infrared-finance","IR","Infrared Finance","DeFi|Liquid staking"),
("initia","INIT","Initia","Layer 1"),
("injective-protocol","INJ","Injective","Layer 1|DeFi"),
("internet-computer","ICP","Internet Computer","Layer 1"),
("intuition","TRUST","Intuition","Infrastructure"),
("iotex","IOTX","IoTeX","Layer 1|Infrastructure"),
("islamic-coin","ISLM","Islamic Coin","Payments"),
("joe","JOE","JOE","DeFi"),
("just","JST","JUST","DeFi"),
("jasmycoin","JASMY","JasmyCoin","Infrastructure"),
("jelly-my-jelly","JELLYJELLY","Jelly-My-Jelly","Meme"),
("jito-governance-token","JTO","Jito","Liquid staking"),
("joe-coin","JOE","Joe Coin","DeFi"),
("jupiter-exchange-solana","JUP","Jupiter","DeFi|Exchange token"),
("chill-guy","CHILLGUY","Just a chill guy","Meme"),
("juventus-fan-token","JUV","Juventus Fan Token",""),
("kaito","KAITO","KAITO","AI"),
("kgen","KGEN","KGeN","Gaming"),
("kaia","KAIA","Kaia","Layer 1"),
("kamino","KMNO","Kamino","DeFi"),
("kaspa","KAS","Kaspa","Layer 1"),
("kava","KAVA","Kava","Layer 1|DeFi"),
("keep-network","KEEP","Keep Network","Infrastructure|Privacy"),
("keeta","KTA","Keeta","Payments"),
("kernel-2","KERNEL","KernelDAO","Liquid staking"),
("kite-2","KITE","Kite","DeFi"),
("klever","KLV","Klever","Payments"),
("koma-inu","KOMA","Koma Inu","Meme"),
("kucoin-shares","KCS","KuCoin","Exchange token"),
("kusama","KSM","Kusama","Layer 1"),
("kyber-network-crystal","KNC","Kyber Network Crystal","DeFi"),
("lcx","LCX","LCX","Exchange token|Real-world assets"),
("lukso-token-2","LYX","LUKSO","Layer 1"),
("lagrange","LA","Lagrange","Infrastructure"),
("lava-network","LAVA","Lava Network","Infrastructure|Oracle"),
("layer3","L3","Layer3","Infrastructure"),
("layerzero","ZRO","LayerZero","Infrastructure"),
("lazio-fan-token","LAZIO","Lazio Fan Token",""),
("frax","FRAX","Legacy Frax Dollar","Payments|DeFi"),
("lido-dao","LDO","Lido DAO","Liquid staking"),
("lighter","LIT","Lighter","DeFi"),
("limewire-token","LMWR","LimeWire",""),
("linea","LINEA","Linea","Layer 2"),
("liquity","LQTY","Liquity","DeFi"),
("lisk","LSK","Lisk","Layer 2"),
("lista","LISTA","Lista DAO","DeFi|Liquid staking"),
("litecoin","LTC","Litecoin","Layer 1|Payments"),
("livepeer","LPT","Livepeer","Infrastructure"),
("loaded-lions","LION","Loaded Lions","Gaming"),
("lombard-protocol","BARD","Lombard","Liquid staking"),
("loopring","LRC","Loopring","Layer 2|DeFi"),
("lorenzo-protocol","BANK","Lorenzo Protocol","Liquid staking|DeFi"),
("lumia","LUMIA","Lumia","Layer 2"),
("luna-by-virtuals","LUNA","Luna by Virtuals","AI"),
("magic-internet-money-runes","MIM","MAGIC INTERNET MONEY (Bitcoin)","DeFi"),
("mantra-dao","OM","MANTRA [Old]","DeFi|Real-world assets"),
("marcopolo","MAPO","MAP Protocol","Layer 2"),
("myx-finance","MYX","MYX Finance","DeFi"),
("magic-eden","ME","Magic Eden","Infrastructure"),
("manchester-city-fan-token","CITY","Manchester City Fan Token",""),
("mango-markets","MNGO","Mango","DeFi"),
("manta-network","MANTA","Manta Network","Layer 2|Privacy"),
("mantle","MNT","Mantle","Layer 2"),
("syrup","SYRUP","Maple Finance","DeFi|Real-world assets"),
("marinade","MNDE","Marinade","Liquid staking"),
("marlin","POND","Marlin","Infrastructure"),
("mask-network","MASK","Mask Network","Infrastructure"),
("maverick-protocol","MAV","Maverick Protocol","DeFi"),
("melania-meme","MELANIA","Melania Meme","Meme"),
("memecore","M","MemeCore","Meme|Layer 1"),
("memecoin-2","MEME","Memecoin","Meme"),
("merlin-chain","MERL","Merlin Chain","Layer 2"),
("metal","MTL","Metal DAO","Payments"),
("meteora","MET","Meteora","DeFi"),
("metis-token","METIS","Metis","Layer 2"),
("milk-alliance","MLK","MiL.k","Real-world assets"),
("milady-meme-coin","LADYS","Milady Meme Coin","Meme"),
("mina-protocol","MINA","Mina Protocol","Layer 1|Privacy"),
("mira-3","MIRA","Mira","DeFi|AI"),
("mitosis","MITO","Mitosis","DeFi|Infrastructure"),
("mobox","MBOX","Mobox","Gaming"),
("mocaverse","MOCA","Moca Network","Gaming"),
("mog-coin","MOG","Mog Coin","Meme"),
("momentum-3","MMT","Momentum","DeFi"),
("monad","MON","Monad","Layer 1"),
("monero","XMR","Monero","Privacy|Payments|Layer 1"),
("moo-deng","MOODENG","Moo Deng","Meme"),
("moonbeam","GLMR","Moonbeam","Layer 1"),
("moonriver","MOVR","Moonriver","Layer 1"),
("moonwell-artemis","WELL","Moonwell","DeFi"),
("morpho","MORPHO","Morpho","DeFi"),
("movement","MOVE","Movement","Layer 1"),
("moviebloc","MBL","MovieBloc","Infrastructure"),
("mubarak","MUBARAK","Mubarak","Meme"),
("elrond-erd-2","EGLD","MultiversX","Layer 1"),
("my-neighbor-alice","ALICE","My Neighbor Alice","Gaming"),
("my-paqman-coin","MPC","My Paqman Coin","Gaming|Meme"),
("myshell","SHELL","MyShell","AI"),
("navi","NAVX","NAVI Protocol","DeFi"),
("near","NEAR","NEAR Protocol","Layer 1"),
("neo","NEO","NEO","Layer 1"),
("nexo","NEXO","NEXO","DeFi"),
("nfprompt-token","NFP","NFPrompt","AI"),
("nkn","NKN","NKN","Infrastructure"),
("nano","XNO","Nano","Payments"),
("neiro-3","NEIRO","Neiro","Meme"),
("neon","NEON","Neon","Layer 2"),
("nervos-network","CKB","Nervos Network","Layer 1"),
("newton-protocol","NEWT","Newton Protocol","Layer 1"),
("nexpace","NXPC","Nexpace","Gaming"),
("nillion","NIL","Nillion","Privacy|Infrastructure"),
("nomina","NOM","Nomina","Layer 1"),
("non-playable-coin","NPC","Non-Playable Coin","Gaming|Meme"),
("nosana","NOS","Nosana","AI|Infrastructure"),
("notcoin","NOT","Notcoin","Gaming|Meme"),
("numeraire","NMR","Numeraire","AI|DeFi"),
("nym","NYM","Nym","Privacy|Infrastructure"),
("og-fan-token","OG","OG Fan Token",""),
("omisego","OMG","OMG Network","Payments|Layer 2"),
("oort","OORT","OORT","Infrastructure|AI"),
("ordinals","ORDI","ORDI","Layer 1"),
("overtake","TAKE","OVERTAKE","Gaming"),
("oasis-network","ROSE","Oasis","Privacy|Layer 1"),
("ocean-protocol","OCEAN","Ocean Protocol","AI"),
("official-trump","TRUMP","Official Trump","Meme"),
("olaxbt","AIO","OlaXBT","AI"),
("omni-network","OMNI","Omni Network [Old]","Infrastructure|Layer 2"),
("ondo-finance","ONDO","Ondo","Real-world assets|DeFi"),
("ontology","ONT","Ontology","Layer 1"),
("ong","ONG","Ontology Gas","Layer 1"),
("chain-2","XCN","Onyxcoin","Layer 1"),
("edu-coin","EDU","Open Campus","Infrastructure"),
("openeden","EDEN","OpenEden","Real-world assets|DeFi"),
("openledger-2","OPEN","OpenLedger","Infrastructure"),
("optimism","OP","Optimism","Layer 2"),
("oraichain-token","ORAI","Oraichain","AI|Oracle"),
("orbs","ORBS","Orbs","Infrastructure|Layer 2"),
("orca","ORCA","Orca","DeFi"),
("orderly-network","ORDER","Orderly","DeFi"),
("origin-protocol","OGN","Origin Token","DeFi"),
("origintrail","TRAC","OriginTrail","Infrastructure|Real-world assets"),
("osmosis","OSMO","Osmosis","DeFi|Layer 1"),
("the-doge-nft","DOG","Own The Doge","Meme"),
("p2p-protocol","P2P","P2P Protocol","Infrastructure"),
("paal-ai","PAAL","PAAL AI","AI"),
("pax-gold","PAXG","PAX Gold","Real-world assets"),
("pha","PHA","PHALA","Privacy|Infrastructure"),
("pivx","PIVX","PIVX","Privacy|Payments"),
("polygon-ecosystem-token","POL","POL (ex-MATIC)","Layer 2"),
("ponke","PONKE","PONKE","Meme"),
("hastra-prime","PRIME","PRIME","DeFi"),
("pancakeswap-token","CAKE","PancakeSwap","DeFi|Exchange token"),
("parcl","PRCL","Parcl","Real-world assets|DeFi"),
("paris-saint-germain-fan-token","PSG","Paris Saint-Germain Fan Token",""),
("particle-network","PARTI","Particle Network","Infrastructure"),
("paxos-standard","USDP","Pax Dollar","Payments"),
("paypal-usd","PYUSD","PayPal USD","Payments"),
("peanut-the-squirrel","PNUT","Peanut the Squirrel","Meme"),
("pendle","PENDLE","Pendle","DeFi"),
("pepe","PEPE","Pepe","Meme"),
("pepecoin-2","PEPECOIN","PepeCoin","Meme"),
("pepecoin-network","PEP","Pepecoin","Meme"),
("phoenix-global","PHB","Phoenix","Infrastructure"),
("red-pulse","PHB","Phoenix Global [OLD]","Infrastructure"),
("pixels","PIXEL","Pixels","Gaming"),
("plasma","XPL","Plasma","Layer 2"),
("playsout","PLAY","PlaysOut","Gaming"),
("plume","PLUME","Plume","Real-world assets|Layer 1"),
("pocket-network","POKT","Pocket Network","Infrastructure"),
("polkadot","DOT","Polkadot","Layer 1"),
("polyhedra-network","ZKJ","Polyhedra Network","Infrastructure"),
("polymesh","POLYX","Polymesh","Real-world assets|Layer 1"),
("popcat","POPCAT","Popcat","Meme"),
("portal-2","PORTAL","Portal","Gaming|Infrastructure"),
("power-ledger","POWR","Powerledger","Real-world assets"),
("prometeus","PROM","Prom","Infrastructure"),
("propy","PRO","Propy","Real-world assets"),
("pudgy-penguins","PENGU","Pudgy Penguins","Meme|Gaming"),
("puffer-finance","PUFFER","Puffer","Liquid staking"),
("pump-fun","PUMP","Pump.fun","DeFi|Meme"),
("pundi-x-2","PUNDIX","Pundi X","Payments"),
("purr-2","PURR","Purr","Meme"),
("pyth-network","PYTH","Pyth Network","Oracle"),
("qtum","QTUM","Qtum","Layer 1"),
("quack-ai","Q","Quack AI","AI|Meme"),
("quant-network","QNT","Quant","Infrastructure"),
("quark-chain","QKC","QuarkChain","Layer 1"),
("quickswap","QUICK","Quickswap","DeFi|Exchange token"),
("ready","READY","READY!","Gaming"),
("roll-2","ROLL","ROLL","Infrastructure"),
("radio-caca","RACA","Radio Caca","Gaming"),
("radix","XRD","Radix","Layer 1"),
("radicle","RAD","Radworks","Infrastructure"),
("rain","RAIN","Rain","Payments"),
("ravedao","RAVE","RaveDAO",""),
("ravencoin","RVN","Ravencoin","Layer 1"),
("raydium","RAY","Raydium","DeFi"),
("reactive-network","REACT","Reactive Network","Infrastructure"),
("redstone-oracles","RED","RedStone","Oracle"),
("unit-00-rei","REI","Rei","Layer 1"),
("rekt-4","REKT","Rekt","Meme"),
("render-token","RENDER","Render","AI|Infrastructure"),
("renzo","REZ","Renzo","Liquid staking"),
("request-network","REQ","Request","Payments|Infrastructure"),
("researchcoin","RSC","ResearchCoin","AI"),
("reserve-rights-token","RSR","Reserve Rights","Payments|DeFi"),
("resolv","RESOLV","Resolv","DeFi"),
("ring-usd","USDR","Ring USD","Payments|DeFi"),
("ripple-usd","RLUSD","Ripple USD","Payments|Real-world assets"),
("river","RIVER","River","Infrastructure"),
("rocket-pool","RPL","Rocket Pool","Liquid staking"),
("rif-token","RIF","Rootstock Infrastructure Framework","Infrastructure"),
("rujira","RUJI","Rujira","DeFi"),
("safecoin","SAFE","SAFEbit","Payments"),
("sats-ordinals","SATS","SATS (Ordinals)","Layer 1"),
("skale","SKL","SKALE","Layer 2"),
("soon-2","SOON","SOON","Layer 2"),
("space-id","ID","SPACE ID","Infrastructure"),
("spx6900","SPX","SPX6900","Meme"),
("subsquid","SQD","SQD","Infrastructure"),
("ssv-network","SSV","SSV Network","Infrastructure|Liquid staking"),
("stbl","STBL","STBL","DeFi|Payments"),
("swftcoin","SWFTC","SWFTCOIN","Payments"),
("saakuru-labs","SKR","Saakuru","Layer 2"),
("safe","SAFE","Safe","Infrastructure"),
("safepal","SFP","SafePal","Infrastructure"),
("saga-2","SAGA","Saga","Layer 1"),
("sahara-ai","SAHARA","Sahara AI","AI"),
("santos-fc-fan-token","SANTOS","Santos FC Fan Token",""),
("sapien-2","SAPIEN","Sapien","AI"),
("scroll","SCR","Scroll","Layer 2"),
("secret","SCRT","Secret","Privacy|Layer 1"),
("seeker","SKR","Seeker",""),
("sei-network","SEI","Sei","Layer 1"),
("sentient","SENT","Sentient","AI"),
("certik","CTK","Shentu","Infrastructure"),
("shiba-inu","SHIB","Shiba Inu","Meme"),
("siacoin","SC","Siacoin","Infrastructure"),
("sideshift-token","XAI","SideShift","DeFi|Exchange token"),
("sign-global","SIGN","Sign","Infrastructure"),
("simon-s-cat","CAT","Simon's Cat","Meme|Gaming"),
("singularitynet","AGIX","SingularityNET","AI"),
("siren-2","SIREN","Siren","DeFi"),
("sky","SKY","Sky","DeFi"),
("smooth-love-potion","SLP","Smooth Love Potion","Gaming"),
("snek","SNEK","Snek","Meme"),
("sosovalue","SOSO","SoSoValue","AI|DeFi"),
("solana","SOL","Solana","Layer 1"),
("solayer","LAYER","Solayer","Liquid staking"),
("solidus-aitech","AITECH","Solidus Ai Tech","AI|Infrastructure"),
("solv-protocol","SOLV","Solv Protocol","Liquid staking|DeFi"),
("somnia","SOMI","Somnia","Layer 1|Gaming"),
("songbird","SGB","Songbird","Layer 1"),
("sonic-3","S","Sonic","Layer 1"),
("sonic-svm","SONIC","Sonic SVM","Layer 1|Gaming"),
("sophon","SOPH","Sophon","Layer 2"),
("space-and-time","SXT","Space and Time","Infrastructure|AI"),
("spacecoin-2","SPACE","Spacecoin","Layer 1"),
("spark-2","SPK","Spark","DeFi"),
("spell-token","SPELL","Spell","DeFi"),
("football-fun","FUN","Sport.fun","Gaming"),
("stablr-euro","EURR","StablR Euro","Payments|Real-world assets"),
("blockstack","STX","Stacks","Layer 2"),
("stader","SD","Stader","Liquid staking"),
("stakestone","STO","StakeStone","Liquid staking"),
("stargate-finance","STG","Stargate Finance","DeFi|Infrastructure"),
("starknet","STRK","Starknet","Layer 2"),
("status","SNT","Status","Infrastructure"),
("steem","STEEM","Steem","Layer 1"),
("stellar","XLM","Stellar","Layer 1|Payments"),
("storj","STORJ","Storj","Infrastructure"),
("story-2","IP","Story","Layer 1"),
("straitsx-xusd","XUSD","StraitsX XUSD","Payments"),
("stronghold-token","SHX","Stronghold","Payments"),
("succinct","PROVE","Succinct","Infrastructure"),
("sui","SUI","Sui","Layer 1"),
("sun-token","SUN","Sun Token","DeFi"),
("superrare","RARE","SuperRare",""),
("superfarm","SUPER","SuperVerse","Gaming"),
("superform","UP","Superform","DeFi"),
("supra","SUPRA","Supra","Oracle|Layer 1"),
("sushi","SUSHI","Sushi","DeFi"),
("swarms","SWARMS","Swarms","AI"),
("synfutures","F","SynFutures","DeFi"),
("synapse-2","SYN","Synapse","Infrastructure|Layer 2"),
("syndicate-3","SYND","Syndicate","Infrastructure"),
("havven","SNX","Synthetix","DeFi"),
("nusd","SUSD","Synthetix sUSD","DeFi|Payments"),
("syscoin","SYS","Syscoin","Layer 1"),
("tac","TAC","TAC","Layer 2"),
("tars-protocol","TAI","TARS AI","AI"),
("thorchain","RUNE","THORChain","DeFi"),
("tria","TRIA","TRIA","Infrastructure"),
("tron","TRX","TRON","Layer 1"),
("taiko","TAIKO","Taiko","Layer 2"),
("talus","US","Talus","AI"),
("telcoin","TEL","Telcoin","Payments"),
("tellor","TRB","Tellor Tributes","Oracle"),
("tensor","TNSR","Tensor","DeFi"),
("terra-luna-2","LUNA","Terra","Layer 1"),
("terra-luna","LUNC","Terra Luna Classic","Layer 1"),
("terrausd","USTC","TerraClassicUSD","Payments"),
("test-3","TST","Test",""),
("tether","USDT","Tether","Payments"),
("tether-gold","XAUT","Tether Gold","Real-world assets|Payments"),
("tezos","XTZ","Tezos","Layer 1"),
("the-graph","GRT","The Graph","Infrastructure|Oracle"),
("the-sandbox","SAND","The Sandbox","Gaming"),
("thena","THE","Thena","DeFi"),
("theta-fuel","TFUEL","Theta Fuel","Infrastructure"),
("theta-token","THETA","Theta Network","Infrastructure"),
("threshold-network-token","T","Threshold Network","Infrastructure|Privacy"),
("thunder-token","TT","ThunderCore","Layer 1"),
("tokamak-network","TON","Tokamak Network","Layer 2"),
("tokenfi","TOKEN","TokenFi","Real-world assets"),
("tokenised-gbp","TGBP","Tokenised GBP","Real-world assets|Payments"),
("the-open-network","TON","Toncoin","Layer 1"),
("tornado-cash","TORN","Tornado Cash","Privacy|DeFi"),
("toshi","TOSHI","Toshi","Meme"),
("towns","TOWNS","Towns","Infrastructure"),
("magic","MAGIC","Treasure","Gaming"),
("tree-capital","TREE","Tree","DeFi"),
("treehouse","TREE","Treehouse","DeFi|Liquid staking"),
("tribe-2","TRIBE","Tribe","DeFi"),
("truefi","TRU","TrueFi","DeFi|Real-world assets"),
("true-usd","TUSD","TrueUSD","Payments"),
("trust-wallet-token","TWT","Trust Wallet","Infrastructure"),
("turbo","TURBO","Turbo","Meme|AI"),
("turtle-4","TURTLE","Turtle","Meme"),
("tutorial","TUT","Tutorial",""),
("uma","UMA","UMA","Oracle|DeFi"),
("usa","USAT","USAT",""),
("usd1-wlfi","USD1","USD1","Payments"),
("usd-coin","USDC","USDC","Payments"),
("usdd","USDD","USDD","Payments"),
("usds","USDS","USDS","Payments"),
("unicorn-fart-dust","UFD","Unicorn Fart Dust","Meme"),
("unifai-network","UAI","UnifAI Network","AI"),
("uniswap","UNI","Uniswap","DeFi"),
("unitas","UP","Unitas","Payments"),
("united-stables","U","United Stables","Payments|DeFi"),
("useless-3","USELESS","Useless Coin","Meme"),
("usual","USUAL","Usual","DeFi"),
("vana","VANA","Vana","AI"),
("vanar-chain","VANRY","Vanar Chain","Layer 1|Gaming"),
("vaulta","A","Vaulta","Layer 1"),
("vechain","VET","VeChain","Layer 1|Real-world assets"),
("vethor-token","VTHO","VeThor","Layer 1"),
("velo","VELO","Velo","Payments|DeFi"),
("velodrome-finance","VELO","Velodrome Finance","DeFi"),
("velvet","VELVET","Velvet","DeFi"),
("venice-token","VVV","Venice Token","AI|Privacy"),
("venom","VENOM","Venom","Layer 1"),
("venus","XVS","Venus","DeFi"),
("verge","XVG","Verge","Privacy|Payments"),
("tomochain","VIC","Viction","Layer 1"),
("victoria-vr","VR","Victoria VR","Gaming"),
("vine","VINE","Vine","Meme"),
("virtual-protocol","VIRTUAL","Virtuals Protocol","AI"),
("vision-3","VSN","Vision",""),
("vulcan-forged","PYR","Vulcan Forged","Gaming"),
("vultisig","VULT","Vultisig","Infrastructure"),
("wax","WAXP","WAX","Gaming|Layer 1"),
("wemix-token","WEMIX","WEMIX","Gaming|Layer 1"),
("wink","WIN","WINkLink","Gaming"),
("woo-network","WOO","WOO","DeFi|Exchange token"),
("wyde-end-hunger","EAT","WYDE: End Hunger","Real-world assets"),
("connect-token-wct","WCT","WalletConnect Token","Infrastructure"),
("walrus-2","WAL","Walrus","Infrastructure"),
("wanchain","WAN","Wanchain","Layer 2|Privacy"),
("waves","WAVES","Waves","Layer 1"),
("wazirx","WRX","WazirX","Exchange token"),
("wilder-world","WILD","Wilder World","Gaming"),
("win-3","WIN","Win","Gaming"),
("world-liberty-financial","WLFI","World Liberty Financial","DeFi"),
("world-mobile-token","WMTX","World Mobile Token","Real-world assets|Infrastructure"),
("worldcoin-wld","WLD","Worldcoin","AI|Infrastructure"),
("wormhole","W","Wormhole","Infrastructure"),
("xdce-crowd-sale","XDC","XDC Network","Layer 1|Real-world assets"),
("xion-2","XION","XION","Layer 1"),
("proton","XPR","XPR Network","Layer 1"),
("ripple","XRP","XRP","Payments|Layer 1"),
("xtcom-token","XT","XT.com","Exchange token"),
("xyo-network","XYO","XYO Network","Oracle|Infrastructure"),
("xai-blockchain","XAI","Xai","Gaming|Layer 2"),
("stratis","STRAX","Xertra","Layer 1"),
("yield-basis","YB","Yield Basis","DeFi"),
("yield-guild-games","YGG","Yield Guild Games","Gaming"),
("yooldo-games","ESPORTS","Yooldo Games","Gaming"),
("zerobase","ZBT","ZEROBASE","Infrastructure"),
("zignaly","ZIG","ZIGChain","DeFi"),
("zksync","ZK","ZKsync","Layer 2"),
("zama","ZAMA","Zama","Privacy|Infrastructure"),
("zcash","ZEC","Zcash","Privacy|Layer 1|Payments"),
("zebec-network","ZBCN","Zebec Network","Payments|Infrastructure"),
("zerebro","ZEREBRO","Zerebro","AI|Meme"),
("zetachain","ZETA","ZetaChain","Layer 1|Infrastructure"),
("zilliqa","ZIL","Zilliqa","Layer 1"),
("apriori","APR","aPriori","Liquid staking"),
("aelf","ELF","aelf","Layer 1"),
("aixbt","AIXBT","aixbt","AI"),
("aura-on-sol","AURA","aura","Liquid staking"),
("cat-in-a-dogs-world","MEW","cat in a dogs world","Meme"),
("dydx-chain","DYDX","dYdX","DeFi|Exchange token"),
("debridge","DBR","deBridge","Infrastructure"),
("dogwifcoin","WIF","dogwifhat","Meme"),
("ecash","XEC","eCash","Payments|Layer 1"),
("edgex","EDGE","edgeX","DeFi"),
("adex","ADX","heyAura",""),
("iexec-rlc","RLC","iExec RLC","Infrastructure|AI"),
("io","IO","io.net","AI|Infrastructure"),
("peaq-2","PEAQ","peaq","Layer 1|Infrastructure"),
("saffron-finance","SFI","saffron.finance","DeFi"),
("yearn-finance","YFI","yearn.finance","DeFi"),
("zkpass","ZKP","zkPass","Privacy|Infrastructure"),
("stable-2","STABLE","Stable","Payments"),
("bianrensheng","BinanceLife","BinanceLife","Meme"),
]

with open('crypto_categories.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['coin_id', 'symbol', 'name', 'categories'])
    for row in data:
        writer.writerow(row)

print(f"Written {len(data)} rows")

Written 715 rows
